# 03 · FunnyBirds + MCBM — does minimality fix grounding?  *(seed-aware)*

**Claim (MCBM):** IB makes `z_j` a minimal sufficient statistic of `c_j`. Loss
`L = CE + λ_c·BCE(z,c) + γ·0.2·mean((6c−3 − z)²)`. **Hypothesis:** minimality constrains
*content*, backwash is about *source*; when `c=f(class)` the ±3 target is class-derived,
so tightening γ can't remove class-reading. Decision rule + null criteria: `DECISIONS §D.5`.
All cells aggregate over seeds. *Refs: `fb_mcbm_renderer_swap.ipynb`, `fb_mcbm_rl_renderer_swap.ipynb`.*

In [ ]:
import os, json, re, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CURATED = Path(os.environ["CURATED_DATA"]); REPO = Path.cwd().parent
import sys; sys.path.insert(0, str(REPO/"analysis"))
try:
    from plotting import set_paper_style, PALETTE; set_paper_style()
    CBM_C, MCBM_C = PALETTE["CBM"], PALETTE["MCBM"]
except Exception:
    CBM_C, MCBM_C = "#0072B2", "#D55E00"
plt.rcParams["figure.dpi"]=120
EPS = 1e-3
def parse_stem(stem):
    m=re.match(r"^funnybirds-(vanilla|cbm|mcbm)(?:-g([0-9p]+))?-s(\d+)$", stem)
    if not m: return None
    gamma=float(m.group(2).replace("p",".")) if m.group(2) else np.nan
    return m.group(1), gamma, int(m.group(3))
def need(p, how):
    ok=Path(p).exists()
    if not ok: print(f"[pending] {p}\n  produce it:  {how}")
    return ok
# ---- seed-aware grounding loaders ----
def load_grounding(prefix):
    """All seeds for a config prefix -> one df with a 'seed' column (None if absent)."""
    fs = sorted(glob.glob(str(CURATED/"grounding"/f"{prefix}-s*.parquet")))
    if not fs: return None
    out=[]
    for f in fs:
        d=pd.read_parquet(f); d["seed"]=int(re.search(r"-s(\d+)\.parquet$", f).group(1)); out.append(d)
    return pd.concat(out, ignore_index=True)
def per_part_seedagg(df, visible_only=True):
    """per (seed,part) retained_frac -> per-part mean/std/count across seeds."""
    d = df[df["changed_frac"]>EPS] if (visible_only and "changed_frac" in df.columns) else df
    g = d.groupby(["seed","part"]).agg(pi=("p_intact","mean"), pr=("p_removed","mean"))
    g["rf"] = g.pr/g.pi
    return g["rf"].groupby("part").agg(["mean","std","count"])


## 1 · Overall `retained_frac` vs γ (±seed std) — the summary (diluted)
`backwash_vs_gamma.csv` (collected on **visible-only** rows). Averages all 5 parts, so
sits near 0.1 regardless — §2 is the one to show. Error bars = std across seeds.

In [ ]:
bw = CURATED/"backwash_vs_gamma.csv"
if need(bw, 'bash analysis/grounding_sweep.sh'):
    T = pd.read_csv(bw); display(T.round(3))
    mc = T[(T.model=="mcbm") & T.retained_frac.notna()]; cb = T[T.model=="cbm"]
    g = mc.groupby("gamma").retained_frac.agg(["mean","std"]).reset_index()
    floor=(g.gamma[g.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(figsize=(6.2,4))
    ax.errorbar(g.gamma.replace(0,floor), g["mean"], yerr=g["std"].fillna(0), marker="o", capsize=3, color=MCBM_C, label="MCBM")
    if len(cb): ax.axhline(cb.retained_frac.mean(), ls="--", color=CBM_C, label="CBM (ref)")
    ax.set_xscale("log"); ax.set_xlabel("γ (effective force = γ×0.2)"); ax.set_ylabel("overall retained_frac")
    ax.set_ylim(0,1.02); ax.set_title("Overall removed-part retention vs γ"); ax.legend()
    print("seeds per gamma:", mc.groupby("gamma").seed.nunique().to_dict())

## 2 · Per-part `retained_frac` vs γ — **the figure**  (tail, ±seed band)
Read every seed's grounding parquet, visible-only, break out by part. Does the **tail**
curve come down as γ rises? (Shaded = ±std across seeds.)

In [ ]:
recs=[]
for f in sorted(glob.glob(str(CURATED/"grounding"/"funnybirds-*-s*.parquet"))):
    pr=parse_stem(Path(f).stem)
    if pr is None or pr[0]=="vanilla": continue
    model,gamma,seed=pr; d=pd.read_parquet(f)
    if "changed_frac" in d.columns: d=d[d["changed_frac"]>EPS]
    gg=d.groupby("part").agg(pi=("p_intact","mean"),pr=("p_removed","mean"))
    for part,row in gg.iterrows(): recs.append(dict(model=model,gamma=gamma,seed=seed,part=part,retained_frac=row.pr/row.pi))
P=pd.DataFrame(recs)
if len(P):
    parts=["tail","wing","beak","foot","eye"]
    A=P.groupby(["model","gamma","part"]).retained_frac.agg(["mean","std"]).reset_index()
    mc=A[A.model=="mcbm"]; cbm_pp=A[A.model=="cbm"].set_index("part")["mean"]
    floor=(mc.gamma[mc.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(figsize=(7,4.4)); cmap=plt.cm.viridis(np.linspace(0,0.85,len(parts)))
    for part,col in zip(parts,cmap):
        s=mc[mc.part==part].sort_values("gamma");
        if not len(s): continue
        x=s.gamma.replace(0,floor)
        ax.plot(x,s["mean"],"o-",color=col,lw=3 if part=="tail" else 1.4,label=part+("  ← highest retention" if part=="tail" else ""))
        ax.fill_between(x, s["mean"]-s["std"].fillna(0), s["mean"]+s["std"].fillna(0), color=col, alpha=0.15)
        if part in cbm_pp.index: ax.scatter([floor/1.6],[cbm_pp[part]],marker="*",s=90,color=col,zorder=5)
    ax.set_xscale("log"); ax.set_xlabel("γ (effective force = γ×0.2)"); ax.set_ylabel("retained_frac (visible-only)")
    ax.set_ylim(-0.02,1.02); ax.set_title("Per-part retention vs γ (★=CBM)\nminimality does not bring tail down"); ax.legend(title="part",fontsize=8)
    print("tail retained_frac by γ (mean±std over seeds):")
    display(mc[mc.part=="tail"][["gamma","mean","std"]].sort_values("gamma").round(3))
else: print("[pending] bash analysis/grounding_sweep.sh")

## 3 · Species-code vs γ (seed-averaged) — did the class channel survive?
If `species←c_preds` stays ≈1 as γ grows, minimality compressed the representation
without cutting the class channel.

In [ ]:
rows=[]
for f in sorted(glob.glob(str(CURATED/"species_probe"/"funnybirds-mcbm-g*-s*.json"))):
    m=re.search(r"-g([0-9p]+)-s(\d+)\.json$", Path(f).name)
    if not m: continue
    S=json.loads(Path(f).read_text())
    rows.append(dict(gamma=float(m.group(1).replace("p",".")), seed=int(m.group(2)),
                     c=S["species_from_cpreds"]["acc"], tail=S["species_from_part_cpreds"].get("tail",{}).get("acc",np.nan),
                     chance=S["chance"]))
if rows:
    D=pd.DataFrame(rows); Dg=D.groupby("gamma").agg(c=("c","mean"),tail=("tail","mean"),chance=("chance","first")).reset_index()
    display(Dg.round(3)); floor=(Dg.gamma[Dg.gamma>0].min() or 0.05)/3
    cbj=sorted(glob.glob(str(CURATED/"species_probe"/"funnybirds-cbm-s*.json")))
    fig,ax=plt.subplots(figsize=(6.2,4))
    ax.plot(Dg.gamma.replace(0,floor),Dg.c,"o-",color=MCBM_C,label="species←c_preds")
    ax.plot(Dg.gamma.replace(0,floor),Dg.tail,"s--",color="#5B8C5A",label="species←tail concepts")
    if cbj: ax.axhline(np.mean([json.loads(Path(p).read_text())["species_from_cpreds"]["acc"] for p in cbj]),ls=":",color=CBM_C,label="CBM species←c_preds")
    ax.axhline(Dg.chance.iloc[0],ls=":",color="k",label="chance"); ax.set_xscale("log")
    ax.set_xlabel("γ"); ax.set_ylabel("species recoverable"); ax.set_ylim(0,1.02); ax.legend(fontsize=8)
    ax.set_title("Class channel survives minimality")
else: print("[pending] grounding_sweep.sh runs the probe too")

## 4 · CONTROL — did γ actually tighten the bottleneck? (seed-averaged)
Flat retention only refutes minimality **if γ changed the representation**. Read `z` from
saved predictions: `mean((6c−3 − z)²)` ↓ / `mean|z|` ↑ with γ = γ bit. See `DECISIONS §D.5`.

In [ ]:
import torch
def zstats_seed(cfg, seed):
    pth = REPO/"external"/"minimal_cbm"/"results"/cfg/str(seed)/"predictions"/"epoch_100.pth"
    if not pth.exists(): return None
    d=torch.load(pth, map_location="cpu", weights_only=False); z,cc=d["z"].float(),d["c"].float()
    if not (np.isfinite(z).all() and np.isfinite(cc).all()): return None
    yp=d["y_preds"]; ta=float((yp.argmax(-1)==d["y"]).float().mean()) if yp.ndim>1 else float((yp==d["y"]).float().mean())
    cp=d["c_preds"]; cp=cp[...,0] if cp.ndim==3 else cp
    return float(((6*cc-3-z)**2).mean()), float(z.abs().mean()), ta, float(((cp>=0.5).float()==cc).float().mean())
rows=[]
for g,tag in [(0,"g0"),(0.1,"g0p1"),(0.3,"g0p3"),(1,"g1"),(3,"g3"),(5,"g5")]:
    for seed in range(1,6):
        s=zstats_seed(f"funnybirds-mcbm-{tag}", seed)
        if s: rows.append((g,seed,*s))
if rows:
    D=pd.DataFrame(rows,columns=["gamma","seed","rep_loss","mean_abs_z","task_acc","concept_acc"])
    Dg=D.groupby("gamma").agg(rep_loss=("rep_loss","mean"),mean_abs_z=("mean_abs_z","mean"),
                              task_acc=("task_acc","mean"),concept_acc=("concept_acc","mean"),n_seeds=("seed","nunique")).reset_index()
    display(Dg.round(3)); floor=(Dg.gamma[Dg.gamma>0].min() or 0.05)/3
    fig,ax=plt.subplots(1,2,figsize=(11,3.8))
    ax[0].plot(Dg.gamma.replace(0,floor),Dg.rep_loss,"o-",color=MCBM_C,label="mean (±3−z)²")
    a0=ax[0].twinx(); a0.plot(Dg.gamma.replace(0,floor),Dg.mean_abs_z,"s--",color="#5B8C5A",label="mean|z|")
    ax[0].set_xscale("log"); ax[0].set_xlabel("γ"); ax[0].set_ylabel("minimality term (↓=pinned)"); a0.set_ylabel("mean|z|"); ax[0].set_title("Did γ bite?")
    ax[1].plot(Dg.gamma.replace(0,floor),Dg.task_acc,"o-",label="task"); ax[1].plot(Dg.gamma.replace(0,floor),Dg.concept_acc,"s--",label="concept")
    ax[1].set_xscale("log"); ax[1].set_xlabel("γ"); ax[1].set_ylabel("val acc"); ax[1].set_ylim(0,1.02); ax[1].legend(); ax[1].set_title("Fit vs γ")
    plt.tight_layout()
    moved=(Dg.rep_loss.max()-Dg.rep_loss.min())>0.05*max(Dg.rep_loss.max(),1e-9) or (Dg.mean_abs_z.max()-Dg.mean_abs_z.min())>0.1
    print("VERDICT:", "γ moved the representation -> flat retention is a real refutation" if moved
          else "γ barely moved -> sweep underpowered; WIDEN γ (DECISIONS §D.5)")
else: print("[pending] need results/funnybirds-mcbm-g*/<seed>/predictions/epoch_100.pth")

## Takeaway
If retention is flat/rising in γ while §4 shows γ tightened the rep and §3 shows species
stays recoverable, minimality shaped *content* but left the *class channel* intact.
**Lock only with ≥3 seeds + a bounded-CI (equivalence) statement — see `DECISIONS §D.5`.**

## How `retained_frac` is read as a backwash measurement
No axis is labelled "backwash"; the computed number is
**`retained_frac = P(concept | part removed) / P(concept | intact)`**, on **visible-only**
removals (the part actually left the render). Grounded → collapses to ~0; backwashed →
stays ~1. `retained_frac` is the metric; "concept–class backwash" is the interpretation.